# Random Forest & Ensemble Experiments — Maximizing Fraud Recall

This notebook systematically tests multiple tree-based models and configurations,
applies F1-optimal and F2-optimal threshold tuning to each, and produces a final
comparison table to identify the best model for fraud recall.

**Experiments:**
1. Baseline Random Forest
2. Random Forest + `class_weight='balanced'`
3. SMOTE + Random Forest (multiple sampling ratios)
4. Calibrated Random Forest
5. Extra Trees Classifier
6. Balanced Random Forest (imblearn)

Each experiment reports **default threshold (0.5)**, **F1-optimal**, and **F2-optimal** results.

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    classification_report, precision_recall_curve,
    average_precision_score, recall_score, precision_score, f1_score
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.ensemble import BalancedRandomForestClassifier

print('All imports OK')

## 1. Data Loading & Preprocessing

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f"Dataset shape: {df.shape}")
print(f"Fraud ratio: {df['Class'].mean():.4%}")

new_df = df.copy()
new_df['Amount'] = RobustScaler().fit_transform(new_df['Amount'].to_numpy().reshape(-1, 1))
new_df['Time']   = StandardScaler().fit_transform(new_df[['Time']])
new_df = new_df.sample(frac=1, random_state=42)

train, temp = train_test_split(new_df, test_size=0.2, stratify=new_df['Class'], random_state=42)
test,  val  = train_test_split(temp,   test_size=0.5, stratify=temp['Class'],   random_state=42)

x_train = train.drop(columns=['Class']); y_train = train['Class']
x_test  = test.drop(columns=['Class']);  y_test  = test['Class']
x_val   = val.drop(columns=['Class']);   y_val   = val['Class']

print(f"Train: {x_train.shape}, Val: {x_val.shape}, Test: {x_test.shape}")
print(f"Val fraud count: {y_val.sum()} / {len(y_val)}")

## 2. Shared Evaluation Helper

Sweeps all thresholds on the precision-recall curve and returns metrics
at the **default (0.5)**, **F1-optimal**, and **F2-optimal** operating points.

In [ ]:
results = []

def evaluate(name, y_true, y_proba):
    """Evaluate a model at default, F1-optimal, and F2-optimal thresholds."""
    prec_arr, rec_arr, thr_arr = precision_recall_curve(y_true, y_proba)
    n = len(thr_arr)
    eps = 1e-9

    f1 = np.array([2*prec_arr[j+1]*rec_arr[j+1] / (prec_arr[j+1]+rec_arr[j+1]+eps) for j in range(n)])
    f2 = np.array([5*prec_arr[j+1]*rec_arr[j+1] / (4*prec_arr[j+1]+rec_arr[j+1]+eps) for j in range(n)])

    pr_auc = average_precision_score(y_true, y_proba)

    # Default threshold = 0.5
    pred_def = (y_proba >= 0.5).astype(int)
    results.append({
        'Experiment': name, 'Threshold': 'default (0.5)',
        'Precision': precision_score(y_true, pred_def),
        'Recall': recall_score(y_true, pred_def),
        'F1': f1_score(y_true, pred_def),
        'PR-AUC': pr_auc
    })

    # F1-optimal
    j1 = np.argmax(f1)
    pred_f1 = (y_proba >= thr_arr[j1]).astype(int)
    results.append({
        'Experiment': name, 'Threshold': f'F1-opt ({thr_arr[j1]:.4f})',
        'Precision': prec_arr[j1+1],
        'Recall': rec_arr[j1+1],
        'F1': f1[j1],
        'PR-AUC': pr_auc
    })

    # F2-optimal (favors recall)
    j2 = np.argmax(f2)
    pred_f2 = (y_proba >= thr_arr[j2]).astype(int)
    results.append({
        'Experiment': name, 'Threshold': f'F2-opt ({thr_arr[j2]:.4f})',
        'Precision': prec_arr[j2+1],
        'Recall': rec_arr[j2+1],
        'F1': f1[j2] if j2 < len(f1) else 0,
        'PR-AUC': pr_auc
    })

    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"{'='*60}")
    print(f"PR-AUC: {pr_auc:.4f}")
    print(f"  Default (0.5):  P={precision_score(y_true, pred_def):.4f}  R={recall_score(y_true, pred_def):.4f}  F1={f1_score(y_true, pred_def):.4f}")
    print(f"  F1-optimal:     P={prec_arr[j1+1]:.4f}  R={rec_arr[j1+1]:.4f}  F1={f1[j1]:.4f}  thr={thr_arr[j1]:.4f}")
    print(f"  F2-optimal:     P={prec_arr[j2+1]:.4f}  R={rec_arr[j2+1]:.4f}  F2={f2[j2]:.4f}  thr={thr_arr[j2]:.4f}")

    return pred_f2  # return F2-optimal predictions for further analysis

print('evaluate() helper ready')

## 3. Experiment 1 — Baseline Random Forest

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1
)
rf_base.fit(x_train, y_train)
proba = rf_base.predict_proba(x_val)[:, 1]
evaluate('1. Baseline RF', y_val, proba)

## 4. Experiment 2 — RF + class_weight='balanced'

In [ ]:
rf_bal = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_bal.fit(x_train, y_train)
proba = rf_bal.predict_proba(x_val)[:, 1]
evaluate('2. RF balanced', y_val, proba)

## 5. Experiment 3 — SMOTE + RF (multiple configs)

Tests several SMOTE sampling ratios with and without `class_weight='balanced'`.

In [ ]:
smote_configs = [
    {'sampling_strategy': 0.2, 'k_neighbors': 5, 'class_weight': None,        'label': 'SMOTE 0.2'},
    {'sampling_strategy': 0.3, 'k_neighbors': 5, 'class_weight': None,        'label': 'SMOTE 0.3'},
    {'sampling_strategy': 0.4, 'k_neighbors': 5, 'class_weight': None,        'label': 'SMOTE 0.4'},
    {'sampling_strategy': 0.5, 'k_neighbors': 5, 'class_weight': None,        'label': 'SMOTE 0.5'},
    {'sampling_strategy': 0.3, 'k_neighbors': 5, 'class_weight': 'balanced',  'label': 'SMOTE 0.3 + balanced'},
    {'sampling_strategy': 0.5, 'k_neighbors': 5, 'class_weight': 'balanced',  'label': 'SMOTE 0.5 + balanced'},
]

for cfg in smote_configs:
    pipe = ImbPipeline([
        ('smote', SMOTE(
            sampling_strategy=cfg['sampling_strategy'],
            k_neighbors=cfg['k_neighbors'],
            random_state=42
        )),
        ('rf', RandomForestClassifier(
            n_estimators=300, max_depth=None, min_samples_leaf=2,
            class_weight=cfg['class_weight'],
            random_state=42, n_jobs=-1
        ))
    ])
    pipe.fit(x_train, y_train)
    proba = pipe.predict_proba(x_val)[:, 1]
    evaluate(f'3. {cfg["label"]}', y_val, proba)

## 6. Experiment 4 — Calibrated Random Forest

Probability calibration (sigmoid) can shift the probability distribution
and improve threshold-tuned recall.

In [ ]:
rf_for_cal = RandomForestClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
rf_for_cal.fit(x_train, y_train)

cal_rf = CalibratedClassifierCV(rf_for_cal, method='sigmoid', cv='prefit')
cal_rf.fit(x_val, y_val)

proba = cal_rf.predict_proba(x_val)[:, 1]
evaluate('4. Calibrated RF (balanced)', y_val, proba)

## 7. Experiment 5 — Extra Trees Classifier

ExtraTrees uses random splits instead of best splits, which can
reduce variance and sometimes improve minority-class recall.

In [ ]:
et = ExtraTreesClassifier(
    n_estimators=300, class_weight='balanced',
    random_state=42, n_jobs=-1
)
et.fit(x_train, y_train)
proba = et.predict_proba(x_val)[:, 1]
evaluate('5. ExtraTrees (balanced)', y_val, proba)

## 8. Experiment 6 — Balanced Random Forest (imblearn)

Balanced RF under-samples the majority class in each bootstrap sample,
giving each tree a balanced view of fraud vs. non-fraud.

In [ ]:
brf = BalancedRandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1
)
brf.fit(x_train, y_train)
proba = brf.predict_proba(x_val)[:, 1]
evaluate('6. BalancedRF (imblearn)', y_val, proba)

## 9. Experiment 7 — SMOTE + Extra Trees

In [ ]:
pipe_et = ImbPipeline([
    ('smote', SMOTE(sampling_strategy=0.3, k_neighbors=5, random_state=42)),
    ('et', ExtraTreesClassifier(
        n_estimators=300, class_weight='balanced',
        random_state=42, n_jobs=-1
    ))
])
pipe_et.fit(x_train, y_train)
proba = pipe_et.predict_proba(x_val)[:, 1]
evaluate('7. SMOTE 0.3 + ExtraTrees', y_val, proba)

## 10. Experiment 8 — RF + SMOTE + Hyperparameter Search

Randomized search over SMOTE ratios, RF depth, and leaf params,
scoring by `recall` to directly optimize for fraud detection.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

pipe_search = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('rf', RandomForestClassifier(random_state=42, n_jobs=-1))
])

param_dist = {
    'smote__sampling_strategy': [0.2, 0.3, 0.4, 0.5],
    'smote__k_neighbors': [3, 5, 7],
    'rf__n_estimators': [200, 300, 500],
    'rf__max_depth': [6, 8, 10, None],
    'rf__min_samples_leaf': [1, 2, 4],
    'rf__class_weight': [None, 'balanced'],
}

search = RandomizedSearchCV(
    pipe_search, param_dist,
    n_iter=30, scoring='recall', cv=3,
    random_state=42, n_jobs=-1, verbose=1
)
search.fit(x_train, y_train)

print(f"\nBest CV recall: {search.best_score_:.4f}")
print(f"Best params: {search.best_params_}")

proba = search.best_estimator_.predict_proba(x_val)[:, 1]
evaluate('8. Tuned SMOTE+RF (recall search)', y_val, proba)

## 11. Results Summary

All experiments compared side-by-side, sorted by recall (F2-optimal threshold).

In [ ]:
results_df = pd.DataFrame(results)
results_df['Precision'] = results_df['Precision'].round(4)
results_df['Recall']    = results_df['Recall'].round(4)
results_df['F1']        = results_df['F1'].round(4)
results_df['PR-AUC']    = results_df['PR-AUC'].round(4)

print('=== ALL RESULTS ===')
print(results_df.to_string(index=False))

print('\n\n=== F2-OPTIMAL RESULTS ONLY (sorted by Recall) ===')
f2_df = results_df[results_df['Threshold'].str.startswith('F2')].sort_values('Recall', ascending=False)
print(f2_df.to_string(index=False))

best_row = f2_df.iloc[0]
print(f"\n*** BEST MODEL FOR RECALL: {best_row['Experiment']}")
print(f"    Recall={best_row['Recall']:.4f}  Precision={best_row['Precision']:.4f}  F1={best_row['F1']:.4f}  PR-AUC={best_row['PR-AUC']:.4f}")

In [ ]:
import matplotlib.pyplot as plt

f2_df_sorted = f2_df.sort_values('Recall', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(f2_df_sorted))
bars = ax.barh(y_pos, f2_df_sorted['Recall'], color='steelblue', edgecolor='black')
ax.set_yticks(y_pos)
ax.set_yticklabels(f2_df_sorted['Experiment'], fontsize=9)
ax.set_xlabel('Recall (Fraud Class)', fontsize=11)
ax.set_title('Fraud Recall by Experiment (F2-Optimal Threshold)', fontsize=13)
ax.axvline(x=0.87, color='red', linestyle='--', linewidth=1, label='Target 0.87')
ax.legend()

for bar, val in zip(bars, f2_df_sorted['Recall']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()